# Purpose of this notebook
The purpose is to perform step 1 of the recommended project workflow to check data quality and split once, then lock the test set away

In [1]:
import numpy as np
import pandas as pd

In [2]:
wine_white = pd.read_csv("datasets/winequality-white.csv", delimiter=';')

# Data Quality Check

## Size and data types

In [3]:
wine_white.shape

(4898, 12)

In [4]:
wine_white.dtypes

fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

## Null values
No missing values

In [5]:
wine_white.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

## Duplicate values

In [6]:
wine_white.duplicated().sum()

np.int64(937)

In [7]:
wine_white = wine_white.drop_duplicates().reset_index(drop=True)

In [8]:
wine_white.shape

(3961, 12)

We identify 937 duplicate rows and drop them, leaving us with 3961 rows

## Domain specific checks

### Checking number of negative values in features (expect 0)
Physicochemical features should not have negative values

In [9]:
numeric_cols = wine_white.select_dtypes(include=['number']).columns

# Identify negative values (< 0) per column
negative_counts = (wine_white[numeric_cols] < 0).sum()
negative_counts

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

### Check physiochemical features that should not be 0

In [10]:
# Flag chemically impossible zeros
impossible_zero_cols = ['density', 'pH', 'fixed acidity', 'alcohol']
impossible_zeros = (wine_white[impossible_zero_cols] == 0.0).sum()
print("\n--- Chemically Impossible Zeros ---")
print(impossible_zeros)


--- Chemically Impossible Zeros ---
density          0
pH               0
fixed acidity    0
alcohol          0
dtype: int64


# Split Into Training and Test Set
Use a 80/20 train test split, and stratification for `quality`

In [11]:
from sklearn.model_selection import train_test_split

# Split the entire DataFrame directly (keeps X and y together)
train_df, test_df = train_test_split(
    wine_white,
    test_size=0.20,
    random_state=211,
    stratify=wine_white['quality']  # Preserves target class proportions
)

# Save the train and test DataFrames to CSV files
train_df.to_csv("datasets/winequality-white-train.csv", index=False)
test_df.to_csv("datasets/winequality-white-test.csv", index=False)